# 01 — Monitoring base table

**Plain English:** We turn the cleaned loans into one tidy table with a few extra columns that every later notebook reuses:

- **vintage** — the approval-year cohort (which year the loan was made),
- **is_default** — `True` if the loan was charged off,
- **size_band** — the loan grouped into a dollar-size bucket,
- **months_to_chargeoff** — for defaulted loans, how many months after approval the charge-off happened (charge-off date − approval date),
- **fully_seasoned** — `True` for older vintages whose outcomes are essentially final; recent vintages still have loans that may yet fail.

In [1]:
%matplotlib inline
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option('display.max_columns', 40, 'display.width', 200)
from src.config import load_config, TABLES_DIR
CFG = load_config()

In [2]:
from src.data_loader import load_clean
from src.base_table import build_base_table
loans = load_clean(config=CFG)
base = build_base_table(loans, config=CFG)
print(f'{len(base):,} funded loans, vintages {int(base.vintage.min())}-{int(base.vintage.max())}')

[2026-06-25 13:10:14] [INFO    ] [data_loader] Loading 2 SBA input file(s): ['foia-7a-fy2000-fy2009-asof-260331.csv', 'foia-7a-fy2010-fy2019-asof-260331.csv']


[2026-06-25 13:10:18] [INFO    ] [data_loader]   foia-7a-fy2000-fy2009-asof-260331.csv -> 690333 rows


[2026-06-25 13:10:21] [INFO    ] [data_loader]   foia-7a-fy2010-fy2019-asof-260331.csv -> 545751 rows


[2026-06-25 13:10:21] [INFO    ] [data_loader] Combined raw frame: 1236084 rows


[2026-06-25 13:10:25] [INFO    ] [data_loader] Dropped 149065 never-funded loans (['CANCLD', 'COMMIT']); 1087019 funded loans remain


[2026-06-25 13:10:27] [INFO    ] [base_table] Base table built: 1087019 loans, 20 vintages (2000-2019), 132662 defaults, 9805 problem exposures


1,087,019 funded loans, vintages 2000-2019


### Result: base-table schema + a sample
The derived fields are on the right. We save the first 500 rows as a lightweight, committable sample (the full table is ~1M rows).

In [3]:
cols = ['vintage','grossapproval','size_band','naics_sector','borrstate',
        'loanstatus','is_default','months_to_chargeoff','fully_seasoned']
sample = base[cols].head(500)
sample.to_csv(TABLES_DIR / '01_base_table_sample.csv', index=False)
base[cols].head(10)

,vintage,grossapproval,size_band,naics_sector,borrstate,loanstatus,is_default,months_to_chargeoff,fully_seasoned
0,2002,510000.0,350k-1m,Manufacturing,PA,CHGOFF,True,103.8,True
1,2002,65000.0,50k-150k,Health Care & Social Assistance,NY,P I F,False,NaN,True
2,2002,351000.0,350k-1m,Retail Trade,CA,P I F,False,NaN,True
3,2002,25000.0,<=50k,Manufacturing,MO,P I F,False,NaN,True
4,2002,135000.0,50k-150k,Accommodation & Food Services,FL,CHGOFF,True,57.1,True
5,2002,288000.0,150k-350k,Retail Trade,CA,P I F,False,NaN,True
6,2002,270000.0,150k-350k,Other Services (except Public Admin),CA,P I F,False,NaN,True
7,2002,450000.0,350k-1m,Retail Trade,NJ,P I F,False,NaN,True
8,2002,952000.0,350k-1m,Wholesale Trade,CA,P I F,False,NaN,True
9,2002,630000.0,350k-1m,Transportation & Warehousing,FL,P I F,False,NaN,True


**Read-out:** every downstream view — concentration, charge-off rates, vintage curves, early warning — is a `groupby` on this one table.